<img src="https://github.com/ArchiColab/sitex/blob/main/images/banner.png?raw=1" width="1000"/>

# Simple Service Area

**The question:** which school or health station is "yours", and how many homes does it serve?
The objective of this notebook is to investigate how the facilities divide the ward among
themselves, which of them carry too much, and where a new facility would help most.

Load a set of facilities, load a street network, and draw each facility's service area:
how far it reaches when you walk a given distance along the streets, and which part of
the neighbourhood each facility serves. The result is one GeoPackage for a QGIS layout.

The notebook is optimized for Google Colab. Because it runs in the cloud, it can be used
on machines with limited computing power and in areas with limited bandwidth.

<img src="https://github.com/ArchiColab/sitex/blob/main/images/SimpleAccess_overview.svg?raw=1" width="1000"/>

**Library:** [`sitex`](https://github.com/ArchiColab/sitex)

**Requires:** two layers, plus optionally the buildings, all prepared and edited in a GIS such as
QGIS and saved as GeoPackage. Nothing is downloaded.
- **A point layer with the facilities** (`POI_FILE`), used exactly as it is: every point counts
  and nothing is filtered. To use only some facilities, select them in QGIS and export them
  first. By default `gdf/kindergarten.gpkg`, the kindergartens of
  `03-NA03-Amenity_Isochrones.ipynb`.
- **A line layer with the street network** (`STREETS_FILE`), by default the OSM walk network
  `osm/pleiku_walk.gpkg`. To test a new street, draw it in QGIS in a copy of this layer.
- Optional, to count the buildings each facility serves: the buildings from
  `01-ARCH-Building_Morphology.ipynb` (`gdf/{slug}_buildings_enriched.gpkg`).

**All layers must be in the same, metric CRS** (see the next section).

**What you edit:** only the *Your choices* cell in Section 1: which files to use.

---
## Coordinate system (CRS)

**The street layer and the facility layer (and the buildings, if used) must be in the same CRS, and it must be a metric one** (a projected CRS such as a UTM zone, not longitude/latitude). Distances, walking times and areas are computed in the unit of that CRS.

- The notebook **reads the CRS from the street file** and checks the other files against it. You do not type an EPSG code. It does not reproject: if a file has no CRS, is in degrees, or differs from the streets, it stops and names the file.
- **To fix a file in QGIS:** right-click the layer → *Export → Save Features As…* → choose the CRS of the street layer → save as GeoPackage.
- The results are saved in the same CRS, so lengths and areas are in metres in QGIS.
- The workshop files are in the site's UTM zone (global UTM, not VN-2000). To combine them with official VN-2000 data, first reproject that data to this CRS.

---
## 1. Setup & your choices

<img src="https://github.com/ArchiColab/sitex/blob/main/images/SimpleAccess_Setup.svg?raw=1" width="1000"/>

Run the first cell once, then set your choices in the second:

| Setting | What it does |
|---|---|
| `STREETS_FILE` | The street network: a line layer, a path inside the data folder. |
| `POI_FILE` | The facilities: a point layer, a path inside the data folder. Every point is used. |
| `BUILDINGS_FILE` | Buildings to count in each catchment and service area, a path inside the data folder. `None` skips the count. |
| `DISTANCES_M` | The service distances in metres, walked along the streets. |
| `AREA_SHAPE` | `"concave"` (recommended) draws the outline around every street junction a facility reaches, as tight as possible while it stays one shape and contains every street walked (Dolmajian, 2023). `"convex"` is the smoother convex hull, which also covers corners no street reaches. |

The CRS is not a setting: it is read from the street file, and the other files must match it.

In [1]:
#@title <font color=#1B7192> Configuration </font>  { display-mode: "form" }

# One-time setup, if `sitex` isn't already installed in this kernel:
# %pip install "sitex[network] @ git+https://github.com/ArchiColab/sitex.git"

import sys
import time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print("Running in:", "Google Colab" if IN_COLAB else "Local environment")

if IN_COLAB:
    %pip install "sitex[network] @ git+https://github.com/ArchiColab/sitex.git"

    from google.colab import drive
    drive.mount("/content/gdrive")

    DATA_DIR = Path("/content/gdrive/MyDrive/Colab_Outputs")
    OUTPUT_DIR = Path("/content/gdrive/MyDrive/SiteX_Outputs")

    if not (DATA_DIR / "overture").is_dir():
        raise FileNotFoundError(
            f"{DATA_DIR} not found (or incomplete) in your Google Drive. "
            "Before the workshop: download the workshop data zip and unzip it into "
            "My Drive/Colab_Outputs -- see the workshop setup instructions."
        )
else:
    DATA_DIR = Path("..") / "data"
    OUTPUT_DIR = Path("..") / "outputs"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import geopandas as gpd
from IPython.display import display

from sitex.core.config import slugify
from sitex.network import isochrones as iso

Running in: Local environment


In [2]:
#@title <font color=#1B7192> Your choices </font>  { display-mode: "form" }

# Your input layers, paths inside the data folder. Edit them in QGIS, save as GeoPackage.
# All must be in the same metric CRS (checked in Section 2).
STREETS_FILE = "osm/pleiku_walk.gpkg"     # lines: the street network
POI_FILE = "gdf/kindergarten.gpkg"        # points: the facilities, every point is used
BUILDINGS_FILE = "gdf/pleiku_buildings_enriched.gpkg"   # buildings to count; None = skip

DISTANCES_M = [500, 1000]     # metres walked along the streets
AREA_SHAPE = "concave"        # "concave" (recommended) or "convex"
WALK_SPEED_KMH = 4.8

# Facilities farther than this from any street are set aside, not snapped to a distant one
MAX_SNAP_M = 250

POI_LABEL = Path(POI_FILE).stem.replace("_", " ").capitalize()   # from the file name, used in the output name

---
## 2. Load POIs

<img src="https://github.com/ArchiColab/sitex/blob/main/images/SimpleAccess_pois.svg?raw=1" width="1000"/>

First the CRS of the files is read and compared (see *Coordinate system* above); the notebook
stops here if they differ. The points are then used exactly as they are in the file: nothing is
classified, cleaned or filtered. To use only some of them, select and export them in QGIS first.
Points without a `name` get a numbered one.

In [3]:
#@title <font color=#1B7192> Click to run </font>  { display-mode: "form" }


_t0 = time.time()
print("### 2. Load POIs ###", flush=True)

print("CRS of the input files:")
LOCAL_EPSG = iso.crs_from_files(
    DATA_DIR / STREETS_FILE,
    {"facilities": DATA_DIR / POI_FILE, "buildings": DATA_DIR / BUILDINGS_FILE if BUILDINGS_FILE else None},
)
print(f"Analysis CRS: EPSG:{LOCAL_EPSG}, read from the street file\n")

facilities = iso.load_points(DATA_DIR / POI_FILE, LOCAL_EPSG, label=POI_LABEL)
print(f"{len(facilities)} points loaded from {POI_FILE}")
if facilities.empty:
    print("No points in the file -- check POI_FILE.")

print(f"--- 2. done in {time.time() - _t0:.2f}s ---\n")
facilities[["name", "geometry"]].head(10)

### 2. Load POIs ###
CRS of the input files:
  streets    pleiku_walk.gpkg: EPSG:32649
  facilities kindergarten.gpkg: EPSG:32649
  buildings  pleiku_buildings_enriched.gpkg: EPSG:32649
Analysis CRS: EPSG:32649, read from the street file

26 points loaded from gdf/kindergarten.gpkg
--- 2. done in 0.42s ---



,name,geometry
0,Trường mầm non Tuổi Thơ,POINT (177808.382 1546554.557)
1,Mầm Non Tư Thục Tuổi Hồng,POINT (177317.636 1546922.765)
2,Mầm Non Sơn Ca,POINT (177108.261 1546696.333)
3,Mầm Non Vành Khuyên,POINT (177260.529 1546598.151)
4,Trường Mầm Non Rạng Đông,POINT (177247.874 1546768.701)
5,Lớp MN Hoa Anh Đào - Tx.AyunPa,POINT (177204.495 1546780.329)
6,Trường Mầm Non Hoa Hồng,POINT (176292.34 1546979.965)
7,Trường Mầm Non Hoa Lan,POINT (176198.632 1547217.061)
8,Trường mầm non Tuổi Thơ Pleiku - Gia Lai,POINT (176292.533 1548004.01)
9,Happy House Steiner - Inspired Kindergarten.,POINT (177519.573 1547433.453)


---
## 3. Street network

<img src="https://github.com/ArchiColab/sitex/blob/main/images/SimpleAccess_network.svg?raw=1" width="1000"/>

The street layer is split wherever two lines cross, so a street you drew in QGIS joins the
existing network where it touches it (turn on snapping while drawing). Each point is then
attached to its nearest street junction; points more than `MAX_SNAP_M` from any street are set
aside.

In [4]:
#@title <font color=#1B7192> Click to run </font>  { display-mode: "form" }


_t0 = time.time()
print("### 3. Street network ###", flush=True)

G = iso.network_from_files([DATA_DIR / STREETS_FILE], LOCAL_EPSG, WALK_SPEED_KMH)
print(f"Network: {G.number_of_nodes():,} junctions, {G.number_of_edges() // 2:,} street segments")

facilities_on, facilities_off = iso.snap_facilities(G, facilities, MAX_SNAP_M)
print(f"{len(facilities_on)} points on the network, {len(facilities_off)} set aside (> {MAX_SNAP_M} m from any street)")
if len(facilities_off):
    print(facilities_off[["name", "snap_dist_m"]].to_string(index=False))

print(f"--- 3. done in {time.time() - _t0:.2f}s ---\n")

### 3. Street network ###
Network: 2,687 junctions, 3,364 street segments
25 points on the network, 1 set aside (> 250 m from any street)
               name  snap_dist_m
Trường Mầm Non 19/8        268.1
--- 3. done in 3.69s ---



---
## 4. Service areas and catchments

<img src="https://github.com/ArchiColab/sitex/blob/main/images/SimpleAccess_analysis.svg?raw=1" width="1000"/>

Two questions about the same facilities, both measured along the streets:

| | Question | Result |
|---|---|---|
| **Service area** | How far does each facility reach? | Everything within each distance in `DISTANCES_M`, one area per facility. Areas overlap where facilities are close together. |
| **Catchment** | Which facility serves which part of the neighbourhood? | Every place goes to the facility with the shortest walk. Catchments never overlap. |

**How a catchment is built.** The notebook walks out from all facilities at once, and each street
junction goes to the facility that reaches it first. Each junction gets the small piece of land
closest to it (a Voronoi cell); the pieces are joined per facility and cut to where that
facility's walk ends. Each cut follows only that facility's own streets, which leaves thin
strips between neighbours; gaps up to 150 m wide are closed and given to the facility whose
cell they lie in. Land in no catchment is beyond the largest distance from every facility.

**Reading the map.** A large catchment means one facility serves a big area alone: a new facility
nearby would take part of it. Small catchments mean facilities are crowded together.

**Buildings served.** With `BUILDINGS_FILE` set, every area gets `n_buildings`, the number of
buildings whose centre lies inside it. Buildings stand in for people, since there is no
population data: a block of flats counts the same as a house. In the catchments each building
counts once; in the service areas it can count for several facilities.

**Why along the streets.** A circle ignores rivers, walls and blocks without through streets.
So it does not reflect the distance to reach a facility.

**The shape.** A concave hull around the streets walked (Dolmajian, 2023). Tested on Pleiku's
buildings, it contains about 96% of the buildings that are truly reachable, and few that aren't;
a buffer around the streets did worse at every width from 15 to 100 m.

> Dolmajian, A. (2023). *Creating isochrones: what is the optimal way?* Medium.
> https://medium.com/@arthur.dolmajian/creating-isochrones-what-is-the-optimal-way-dfc77a2ca13a
>

The table lists the five largest catchments. On the map, hover over a catchment for its facility,
area and buildings; switch on the service-area outlines in the layer control.

In [5]:
#@title <font color=#1B7192> Click to run </font>  { display-mode: "form" }


_t0 = time.time()
print("### 4. Service areas and catchments ###", flush=True)

areas = iso.facility_service_areas(G, facilities_on, DISTANCES_M, shape=AREA_SHAPE)
catchments = iso.nearest_facility_catchments(G, facilities_on, max(DISTANCES_M), shape=AREA_SHAPE)

print(f"{len(areas)} service areas ({len(facilities_on)} facilities x {len(DISTANCES_M)} distances), "
      f"{len(catchments)} catchments, {catchments['area_ha'].sum():,.0f} ha in total")

table_cols = ["name", "area_ha"]
if BUILDINGS_FILE:
    buildings = gpd.read_file(DATA_DIR / BUILDINGS_FILE)
    catchments = iso.count_buildings(catchments, buildings)
    areas = iso.count_buildings(areas, buildings)
    served = catchments["n_buildings"].sum()
    print(f"\nBuildings: {len(buildings):,} in the study area, {served:,} in a catchment "
          f"({100 * served / len(buildings):.0f}%), {len(buildings) - served:,} in none "
          f"(roughly more than a {max(DISTANCES_M)} m walk from every {POI_LABEL.lower()})")
    table_cols.append("n_buildings")

print("\nLargest catchments:")
print(catchments.nlargest(5, "area_ha")[table_cols].to_string(index=False))

print(f"--- 4. done in {time.time() - _t0:.2f}s ---\n")
iso.preview_service_map(catchments, areas, facilities_on)

### 4. Service areas and catchments ###
50 service areas (25 facilities x 2 distances), 24 catchments, 551 ha in total

Buildings: 21,389 in the study area, 11,922 in a catchment (56%), 9,467 in none (roughly more than a 1000 m walk from every kindergarten)

Largest catchments:
                                        name  area_ha  n_buildings
                     Trường mầm non Tuổi Thơ     77.1         1387
Happy House Steiner - Inspired Kindergarten.     51.8         1127
                         Nhom Tre Phuong Nhi     51.6         1311
                         Mầm Non Vành Khuyên     45.7         1038
                Trường Mầm Non Hoa Phong Lan     37.2          866
--- 4. done in 18.80s ---



c:\Users\Maddie\anaconda3\envs\gis\lib\site-packages\folium\features.py:1204: UserWarning: GeoJsonTooltip is not configured to render for GeoJson GeometryCollection geometries. Please consider reworking these features: [{'id': '243c54a8-c476-466c-a5d4-137cda8684e1', 'name': 'Mầm Non Tư Thục Tuổi Hồng', 'area_ha': 21.1, 'n_buildings': 585}, {'id': '1e3b107c-fb6a-4e53-a0f0-42ac5d6a9738', 'name': 'Trường mầm non Tuổi Thơ Pleiku - Gia Lai', 'area_ha': 21.0, 'n_buildings': 441}, {'id': '38fd0576-c6aa-4f51-891d-00f8c3080646', 'name': 'Happy House Steiner - Inspired Kindergarten.', 'area_ha': 51.8, 'n_buildings': 1127}, {'id': '587166b0-0f07-4861-957e-9ada4b831ecf', 'name': 'Mầm Non Bé Yêu', 'area_ha': 13.8, 'n_buildings': 152}, {'id': '0b4cc97b-12fd-4b2a-bc5e-ef4375d6abec', 'name': 'Homy Preschool Pleiku', 'area_ha': 16.0, 'n_buildings': 423}, {'id': '40cf1ca9-9bb0-450d-a695-638b59c7aa89', 'name': 'Trường Mầm Non Hoa Phong Lan', 'area_ha': 37.2, 'n_buildings': 866}, {'id': 'e2b3b432-a7c9-420

---
## 5. Export to QGIS

<img src="https://github.com/ArchiColab/sitex/blob/main/images/SimpleAccess_export.svg?raw=1" width="1000"/>

One GeoPackage in `data/gdf/`, in the CRS of your input layers (metres), so lengths and areas are
in metres in QGIS:

| Layer | Contents |
|---|---|
| `catchments` | One area per facility, not overlapping, with `area_ha` and `n_buildings`. Style it as *Categorized* on `name`. |
| `service_areas` | Each facility's reach per `distance_m`, with `area_ha` and `n_buildings`; they overlap. Show one distance with a filter such as `"distance_m" = 500`. |
| `facilities` | The facilities used, with `snap_dist_m`. |
| `network` | The street network the walks ran on. |

In [6]:
#@title <font color=#1B7192> Click to run </font>  { display-mode: "form" }


_t0 = time.time()
print("### 5. Export to QGIS ###", flush=True)

out_path = iso.export_geopackage(
    DATA_DIR / "gdf" / f"service_areas_{slugify(POI_LABEL)}.gpkg",
    {
        "catchments": catchments,
        "service_areas": areas,
        "facilities": facilities_on.drop(columns=["node"]),
        "network": iso.network_edges(G),
    },
)
print(f"Saved {out_path}")

print(f"--- 5. done in {time.time() - _t0:.2f}s ---\n")

### 5. Export to QGIS ###
Saved ..\data\gdf\service_areas_kindergarten.gpkg
--- 5. done in 0.23s ---



---
## 6. Field check: who comes, and how?

The catchment map shows which facility is *nearest*. The field check shows who actually comes,
and how they get there.

**How to do it, with one example:**

1. On the map in Section 4, pick one facility, for example the kindergarten with the largest
   catchment (`n_buildings`).
2. Stand at its entrance for 30 minutes at opening or closing time, and count every arrival by
   how they came: **on foot, by bicycle, by motorbike, by car**.
3. Ask 3–5 people where they came from and why they chose *this* facility.
4. Compare with the map: do they live inside its catchment?

*Example of a record:*

> Kindergarten D, largest catchment (312 buildings). Wednesday, 16:30–17:00.
>
> | On foot | Bicycle | Motorbike | Car | Total |
> |---|---|---|---|---|
> | 3 | 1 | 36 | 2 | 42 |
>
> 5 parents asked: 4 live inside D's catchment, all less than 1 km away, and still came by
> motorbike ("it's hot", "I go on to work"). 1 lives in the next catchment and chose D because
> it is on her way to the market.

For reference: 1 km along the streets is about **12 minutes on foot** or **4 minutes by bike**.
In a 15-minute city (Moreno et al., 2021), trips this short are the ones people could make on
foot or by bike. Your count shows how many actually do.

---
## 7. Test a new service

This notebook can test a proposal: where would a new kindergarten, health station or market help
most? Add the new facility in QGIS and re-run.

1. **Work on a copy.** In QGIS, right-click your facility layer → *Export → Save Features As…* →
   `gdf/kindergarten_proposal.gpkg`, keeping the same CRS. The original stays untouched.
2. **Add the point.** Open the copy, *Toggle Editing*, *Add Point Feature*, and click where the
   new facility would be, close to a street.
3. **Give it a `name`**, for example `New kindergarten (proposal)`; it is shown on the map and in
   the table. Other attributes can stay empty, since every point in the file is used.
4. **Save** the edits, then set `POI_FILE = "gdf/kindergarten_proposal.gpkg"` in Section 1 and
   re-run the notebook.
5. **Compare** with the result before: which catchments shrink, how many buildings the new
   facility takes over (`n_buildings`), and how many buildings are left in no catchment.

To test a new street instead, draw it in a copy of the street layer (snapping on, so it joins the
network) and set `STREETS_FILE` to the copy.

---
## Notes

- **Distances are walked along the streets**, so a facility across a river or a block without
  through streets serves less than a circle on the map would suggest.
- **The edge of each area is approximate.** A hull also covers some corners between streets;
  `"convex"` covers more of them than the default `"concave"`.
- **A catchment is not a service level.** It shows the *nearest* facility, not whether that
  facility has room, the right opening hours or the right services.
- **Facilities outside the study area are missing.** People near the edge may walk to a
  facility just outside it, so catchments along the edge look larger than they are.
- **Buildings are not people.** `n_buildings` counts every building the same, whatever its
  size or use.
- **Two facilities at the same street junction share one catchment** (`n_facilities_here` = 2).
- **To test a new facility**, see Section 7: add it in QGIS to a copy of the point file and
  re-run.

---
## When the model is wrong

Where your notes and the catchments disagree, write down why. Common reasons:

- **Nearest isn't used.** Quality, price, opening hours, habit or the route to work decide which
  facility people choose.
- **Rules draw their own lines.** School admission areas follow the administration, not the
  walk.
- **Buildings are not people,** and a facility's catchment says nothing about its capacity.
- **Facilities outside the study area** take people from its edge.

Write one sentence per disagreement: *"The map says …, I found …, because …"*.

## Where to go next

| Next step | Open |
|---|---|
| Walking time to green space, and where green is nearby but not public | [`03-NA06-Green_Accessibility.ipynb`](03-NA06-Green_Accessibility.ipynb) |
| Test a new facility or street link as walking areas | [`03-NA04-Simple_Isochrone.ipynb`](03-NA04-Simple_Isochrone.ipynb) |

**Into the design phase:** put a candidate facility in the largest catchment or among the
buildings in no catchment, re-run, and compare `n_buildings` before and after. That is the case
for your proposal in one number.